# **ANÁLISIS EXPLORATORIO DE DATOS PARTE 2**

1. Tipos de datos

2. Manejando valores vacíos

3. Renombramiento columnas e índices

4. Combinando dataframes

5. Manejando fechas

<img src="./imgs/foto-dia-03.jpg" width="500px" height="300px">


En esta ejercicio vamos a trabajar con el dataset de canciones en Spotify en 2023, ejecuta la siguiente celda para ver las columnas del dataframe.


In [33]:
import pandas as pd

spotify = pd.read_csv('datasets/spotify-2023.csv', encoding='latin-1')
spotify.head()

,track_name,artist(s)_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,...,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
0,Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2,2023,7,14,553,147,141381703,43,...,125,B,Major,80,89,83,31,0,8,4
1,LALA,Myke Towers,1,2023,3,23,1474,48,133716286,48,...,92,C#,Major,71,61,74,7,0,10,4
2,vampire,Olivia Rodrigo,1,2023,6,30,1397,113,140003974,94,...,138,F,Major,51,32,53,17,0,31,6
3,Cruel Summer,Taylor Swift,1,2019,8,23,7858,100,800840817,116,...,170,A,Major,55,58,72,11,0,11,15
4,WHERE SHE GOES,Bad Bunny,1,2023,5,18,3133,50,303236322,84,...,144,A,Minor,65,23,80,14,63,11,6


### **1. Tipos de datos**


El tipo de datos de una columna en un DataFrame o una Serie se conoce como **dtype**.

Puedes utilizar la propiedad dtype para obtener el tipo de una columna específica. Por ejemplo, podemos obtener el dtype de la columna precio en el DataFrame revisiones:


In [34]:
spotify['artist_count'].dtypes

dtype('int64')

Alternativamente, la propiedad dtypes devuelve el dtype de cada columna del DataFrame:


In [35]:
spotify.dtypes

track_name              object
artist(s)_name          object
artist_count             int64
released_year            int64
released_month           int64
released_day             int64
in_spotify_playlists     int64
in_spotify_charts        int64
streams                 object
in_apple_playlists       int64
in_apple_charts          int64
in_deezer_playlists     object
in_deezer_charts         int64
in_shazam_charts        object
bpm                      int64
key                     object
mode                    object
danceability_%           int64
valence_%                int64
energy_%                 int64
acousticness_%           int64
instrumentalness_%       int64
liveness_%               int64
speechiness_%            int64
dtype: object

Los tipos de datos nos dicen algo sobre cómo pandas está almacenando los datos internamente. float64 significa que está usando un número de coma flotante de 64 bits; int64 significa un entero de tamaño similar, y así sucesivamente.

Una peculiaridad a tener en cuenta (y que se muestra muy claramente aquí) es que las columnas que consisten enteramente en cadenas no tienen su propio tipo; en su lugar se les da el tipo de objeto.

Es posible convertir una columna de un tipo en otro siempre que dicha conversión tenga sentido utilizando la función astype(). Por ejemplo, podemos transformar la columna de `artist_count` de su actual tipo de datos int64 a un tipo de datos float64:


In [36]:
spotify['artist_count'].astype('Float64')

0      2.0
1      1.0
2      1.0
3      1.0
4      1.0
      ... 
948    1.0
949    1.0
950    2.0
951    3.0
952    1.0
Name: artist_count, Length: 953, dtype: Float64

Un índice DataFrame o Series también tiene su propio dtype:


### **2. Manejando valores vacíos**


Los valores que faltan en los registros reciben el valor NaN, abreviatura de "Not a Number" (no es un número). Por razones técnicas estos valores NaN son siempre del tipo float64.

Pandas proporciona algunos métodos específicos para datos perdidos. Para seleccionar registros NaN se puede utilizar pd.isnull() (o su compañera pd.notnull()). Esto está pensado para ser usado así:


In [37]:
spotify.loc[spotify['key'].isnull()].shape

(95, 24)

Para saber que columnas tienen valores nulos dentro de nuestro dataframe podemos utilizar el siguiente código que nos dice el porcentaje de valores nulos para cada columna.


In [38]:
porcetanje_nulos_por_columna = spotify.isnull().sum() / len(spotify) * 100
porcetanje_nulos_por_columna

track_name              0.00000
artist(s)_name          0.00000
artist_count            0.00000
released_year           0.00000
released_month          0.00000
released_day            0.00000
in_spotify_playlists    0.00000
in_spotify_charts       0.00000
streams                 0.00000
in_apple_playlists      0.00000
in_apple_charts         0.00000
in_deezer_playlists     0.00000
in_deezer_charts        0.00000
in_shazam_charts        5.24659
bpm                     0.00000
key                     9.96852
mode                    0.00000
danceability_%          0.00000
valence_%               0.00000
energy_%                0.00000
acousticness_%          0.00000
instrumentalness_%      0.00000
liveness_%              0.00000
speechiness_%           0.00000
dtype: float64

Reemplazar valores NaN es una operación común. Pandas proporciona un método realmente práctico para este problema: `fillna()`.

fillna() proporciona unas cuantas estrategias diferentes para mitigar tales datos. Por ejemplo, podemos simplemente reemplazar cada NaN por un "Desconocido":


In [39]:
spotify['key'] = spotify['key'].fillna('unknown')
spotify['key'].value_counts()

key
C#         120
G           96
unknown     95
G#          91
F           89
B           81
D           81
A           75
F#          73
E           62
A#          57
D#          33
Name: count, dtype: int64

O podemos rellenar cada valor que falte con el primer valor no nulo que aparezca en algún momento después del registro dado en la base de datos. Esto se conoce como estrategia de relleno.

También podemos tener un valor no nulo que queramos sustituir. Por ejemplo, supongamos que desde que se publicó este conjunto de datos, Bad Bunny ha cambiado su nombre artístico a Good Bunny, pues esto lo podríamos reflejar así.


In [40]:
spotify['artist(s)_name'] = spotify['artist(s)_name'].replace('Bad Bunny', 'Good Bunny')
spotify.loc[spotify['artist(s)_name'] == 'Good Bunny']

,track_name,artist(s)_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,...,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
4,WHERE SHE GOES,Good Bunny,1,2023,5,18,3133,50,303236322,84,...,144,A,Minor,65,23,80,14,63,11,6
192,Titi Me Preguntï¿½,Good Bunny,1,2022,5,6,9037,42,1264310836,124,...,107,F,Minor,65,19,72,10,0,13,25
239,Efecto,Good Bunny,1,2022,5,6,4004,33,1047480053,34,...,98,G,Minor,80,23,48,14,0,6,5
362,Neverita,Good Bunny,1,2022,5,6,2590,30,671365962,20,...,122,A#,Major,88,43,50,7,0,14,5
377,Moscow Mule,Good Bunny,1,2022,5,6,4572,33,909001996,74,...,100,F,Minor,80,29,67,29,0,12,3
422,Yonaguni,Good Bunny,1,2021,6,4,9644,28,1260594497,120,...,180,C#,Major,64,44,65,28,0,14,12
676,A Tu Merced,Good Bunny,1,2020,2,29,4214,11,685071800,21,...,92,unknown,Major,86,89,79,17,0,11,6
716,La Zona,Good Bunny,1,2020,2,29,1188,0,312622938,13,...,94,C#,Minor,76,81,80,20,0,25,4
766,Despuï¿½ï¿½s de la P,Good Bunny,1,2022,5,6,2229,0,461558540,27,...,78,F,Major,56,61,90,36,0,18,31
767,Un Ratito,Good Bunny,1,2022,5,6,1112,6,417230415,7,...,93,unknown,Minor,79,22,55,31,0,12,5


### **3. Renombrando columnas e índices**


A menudo los datos nos llegan con nombres de columnas, índices u otras convenciones de nomenclatura con las que no estamos satisfechos. Para eso tenemos las funciones `rename` y `rename_axis`


La primera función rename(), permite cambiar los nombres de índices y/o columnas. Por ejemplo, para cambiar la columna nombre de artistas de nuestro conjunto de datos, haríamos:


In [41]:
spotify.rename(columns={'artist(s)_name': 'artist_name'})

,track_name,artist_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,...,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
0,Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2,2023,7,14,553,147,141381703,43,...,125,B,Major,80,89,83,31,0,8,4
1,LALA,Myke Towers,1,2023,3,23,1474,48,133716286,48,...,92,C#,Major,71,61,74,7,0,10,4
2,vampire,Olivia Rodrigo,1,2023,6,30,1397,113,140003974,94,...,138,F,Major,51,32,53,17,0,31,6
3,Cruel Summer,Taylor Swift,1,2019,8,23,7858,100,800840817,116,...,170,A,Major,55,58,72,11,0,11,15
4,WHERE SHE GOES,Good Bunny,1,2023,5,18,3133,50,303236322,84,...,144,A,Minor,65,23,80,14,63,11,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
948,My Mind & Me,Selena Gomez,1,2022,11,3,953,0,91473363,61,...,144,A,Major,60,24,39,57,0,8,3
949,Bigger Than The Whole Sky,Taylor Swift,1,2022,10,21,1180,0,121871870,4,...,166,F#,Major,42,7,24,83,1,12,6
950,A Veces (feat. Feid),"Feid, Paulo Londra",2,2022,11,3,573,0,73513683,2,...,92,C#,Major,80,81,67,4,0,8,6
951,En La De Ella,"Feid, Sech, Jhayco",3,2022,10,20,1320,0,133895612,29,...,97,C#,Major,82,67,77,8,0,12,5


`rename()` permite renombrar valores de índice o columna especificando un parámetro de palabra clave de índice o columna, respectivamente. Admite varios formatos de entrada, pero normalmente el más conveniente es un diccionario de Python. Un ejemplo que lo utiliza para renombrar algunos elementos del índice es el siguiente:


In [42]:
spotify.rename(index={
    0:10000,
    1:'SecondEntry'
    })

,track_name,artist(s)_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,...,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
10000,Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2,2023,7,14,553,147,141381703,43,...,125,B,Major,80,89,83,31,0,8,4
SecondEntry,LALA,Myke Towers,1,2023,3,23,1474,48,133716286,48,...,92,C#,Major,71,61,74,7,0,10,4
2,vampire,Olivia Rodrigo,1,2023,6,30,1397,113,140003974,94,...,138,F,Major,51,32,53,17,0,31,6
3,Cruel Summer,Taylor Swift,1,2019,8,23,7858,100,800840817,116,...,170,A,Major,55,58,72,11,0,11,15
4,WHERE SHE GOES,Good Bunny,1,2023,5,18,3133,50,303236322,84,...,144,A,Minor,65,23,80,14,63,11,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
948,My Mind & Me,Selena Gomez,1,2022,11,3,953,0,91473363,61,...,144,A,Major,60,24,39,57,0,8,3
949,Bigger Than The Whole Sky,Taylor Swift,1,2022,10,21,1180,0,121871870,4,...,166,F#,Major,42,7,24,83,1,12,6
950,A Veces (feat. Feid),"Feid, Paulo Londra",2,2022,11,3,573,0,73513683,2,...,92,C#,Major,80,81,67,4,0,8,6
951,En La De Ella,"Feid, Sech, Jhayco",3,2022,10,20,1320,0,133895612,29,...,97,C#,Major,82,67,77,8,0,12,5


Probablemente renombrará columnas muy a menudo, pero renombrará valores de índice muy raramente. Para eso, `set_index()` suele ser más conveniente ya que normalmente renombraremos el índice con los valores de alguna columna.

Tanto el índice de fila como el índice de columna pueden tener su propio atributo nombre. El método `rename_axis()` puede utilizarse para cambiar estos nombres. Por ejemplo:


In [43]:
spotify.rename_axis('canciones', axis='rows').rename_axis('nombre_columnas', axis='columns')

nombre_columnas,track_name,artist(s)_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,...,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
canciones,,,,,,,,,,,,,,,,,,,,,
0,Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2,2023,7,14,553,147,141381703,43,...,125,B,Major,80,89,83,31,0,8,4
1,LALA,Myke Towers,1,2023,3,23,1474,48,133716286,48,...,92,C#,Major,71,61,74,7,0,10,4
2,vampire,Olivia Rodrigo,1,2023,6,30,1397,113,140003974,94,...,138,F,Major,51,32,53,17,0,31,6
3,Cruel Summer,Taylor Swift,1,2019,8,23,7858,100,800840817,116,...,170,A,Major,55,58,72,11,0,11,15
4,WHERE SHE GOES,Good Bunny,1,2023,5,18,3133,50,303236322,84,...,144,A,Minor,65,23,80,14,63,11,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
948,My Mind & Me,Selena Gomez,1,2022,11,3,953,0,91473363,61,...,144,A,Major,60,24,39,57,0,8,3
949,Bigger Than The Whole Sky,Taylor Swift,1,2022,10,21,1180,0,121871870,4,...,166,F#,Major,42,7,24,83,1,12,6
950,A Veces (feat. Feid),"Feid, Paulo Londra",2,2022,11,3,573,0,73513683,2,...,92,C#,Major,80,81,67,4,0,8,6


### **4. Combinando dataframes**


Al realizar operaciones en un conjunto de datos, a veces necesitaremos combinar diferentes DataFrames y/o Series de formas no sencillas. Pandas tiene tres métodos básicos para hacer esto. En orden de complejidad creciente, son `concat()`, `join()` y `merge()`. La mayor parte de lo que puede hacer merge() también se puede hacer de forma más sencilla con join(), por lo que la omitiremos y nos centraremos en las dos primeras funciones.

El método de combinación más sencillo es concat(). Dada una lista de elementos, esta función los unirá a lo largo de un eje.

Esto es útil cuando tenemos datos en diferentes objetos DataFrame o Series pero que tienen los mismos campos (columnas). Un ejemplo, es el que vamos a ver ahora con el conjunto de datos YouTube Videos, que divide los datos en función del país de origen (por ejemplo, Canadá y USA, en este ejemplo). Si queremos estudiar varios países a la vez, podemos utilizar concat() para agruparlos.


In [44]:
canadian_youtube = pd.read_csv('./datasets/CAvideos.csv')
usa_youtube = pd.read_csv('./datasets/USvideos.csv')

union_youtube = pd.concat([canadian_youtube, usa_youtube])
union_youtube.shape


(81830, 16)

El combinador intermedio en términos de complejidad es `join()`. join() permite combinar diferentes objetos DataFrame que tienen un índice en común. Por ejemplo, para obtener los vídeos que fueron tendencia el mismo día en Canadá y en USA, podemos hacer lo siguiente:


In [45]:
left = canadian_youtube.set_index(['title', 'trending_date'])
right = usa_youtube.set_index(['title', 'trending_date'])

inner_join = left.join(right, lsuffix='_CAN', rsuffix='_USA', how='inner')

inner_join.head()

,,video_id_CAN,channel_title_CAN,category_id_CAN,publish_time_CAN,tags_CAN,views_CAN,likes_CAN,dislikes_CAN,comment_count_CAN,thumbnail_link_CAN,...,tags_USA,views_USA,likes_USA,dislikes_USA,comment_count_USA,thumbnail_link_USA,comments_disabled_USA,ratings_disabled_USA,video_error_or_removed_USA,description_USA
title,trending_date,,,,,,,,,,,,,,,,,,,,,
Eminem - Walk On Water (Audio) ft. Beyoncé,17.14.11,n1WpP7iowLc,EminemVEVO,10,2017-11-10T17:00:03.000Z,"Eminem|""Walk""|""On""|""Water""|""Aftermath/Shady/In...",17158579,787425,43420,125882,https://i.ytimg.com/vi/n1WpP7iowLc/default.jpg,...,"Eminem|""Walk""|""On""|""Water""|""Aftermath/Shady/In...",17158531,787419,43420,125882,https://i.ytimg.com/vi/n1WpP7iowLc/default.jpg,False,False,False,Eminem's new track Walk on Water ft. Beyoncé i...
"Racist Superman | Rudy Mancuso, King Bach & Lele Pons",17.14.11,5qpjK5DgCt4,Rudy Mancuso,23,2017-11-12T19:05:24.000Z,"racist superman|""rudy""|""mancuso""|""king""|""bach""...",3191434,146035,5339,8181,https://i.ytimg.com/vi/5qpjK5DgCt4/default.jpg,...,"racist superman|""rudy""|""mancuso""|""king""|""bach""...",3191434,146033,5339,8181,https://i.ytimg.com/vi/5qpjK5DgCt4/default.jpg,False,False,False,WATCH MY PREVIOUS VIDEO ▶ \n\nSUBSCRIBE ► http...
I Dare You: GOING BALD!?,17.14.11,d380meD0W0M,nigahiga,24,2017-11-12T18:01:41.000Z,"ryan|""higa""|""higatv""|""nigahiga""|""i dare you""|""...",2095828,132239,1989,17518,https://i.ytimg.com/vi/d380meD0W0M/default.jpg,...,"ryan|""higa""|""higatv""|""nigahiga""|""i dare you""|""...",2095731,132235,1989,17518,https://i.ytimg.com/vi/d380meD0W0M/default.jpg,False,False,False,I know it's been a while since we did this sho...
Ed Sheeran - Perfect (Official Music Video),17.14.11,2Vv-BfVoq4g,Ed Sheeran,10,2017-11-09T11:04:14.000Z,"edsheeran|""ed sheeran""|""acoustic""|""live""|""cove...",33523622,1634130,21082,85067,https://i.ytimg.com/vi/2Vv-BfVoq4g/default.jpg,...,"edsheeran|""ed sheeran""|""acoustic""|""live""|""cove...",33523622,1634124,21082,85067,https://i.ytimg.com/vi/2Vv-BfVoq4g/default.jpg,False,False,False,🎧: https://ad.gt/yt-perfect\n💰: https://atlant...
WE WANT TO TALK ABOUT OUR MARRIAGE,17.14.11,2kyS6SvSYSE,CaseyNeistat,22,2017-11-13T17:13:01.000Z,SHANtell martin,748374,57534,2967,15959,https://i.ytimg.com/vi/2kyS6SvSYSE/default.jpg,...,SHANtell martin,748374,57527,2966,15954,https://i.ytimg.com/vi/2kyS6SvSYSE/default.jpg,False,False,False,SHANTELL'S CHANNEL - https://www.youtube.com/s...


Los parámetros lsuffix y rsuffix son necesarios aquí porque los datos tienen los mismos nombres de columna en los conjuntos de datos británicos y estado unidenses. Si no fuera así (porque, por ejemplo, les hubiéramos cambiado el nombre de antemano) no los necesitaríamos.


### **5. Manejando fechas**


En el análisis de datos, a menudo necesitamos trabajar con datos de fecha y hora. Pandas proporciona numerosas funciones y métodos para manejar eficazmente datos de fecha y hora. A continuación, vamos a ver algunas de las funciones más comunes para el manejo de fechas en Pandas:

La función `pd.to_datetime()` se utiliza para convertir una columna a un tipo de dato de fecha y hora. Esto es útil cuando los datos de fecha y hora están en formato de cadena y necesitas convertirlos a un formato reconocido por Pandas.


In [50]:
canadian_youtube.dtypes

canadian_youtube['publish_time'] = pd.to_datetime(canadian_youtube['publish_time'])
canadian_youtube['trending_date'] = pd.to_datetime(canadian_youtube['trending_date'], 
                                                   format='%y.%d.%m')

canadian_youtube.dtypes
canadian_youtube.head()

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description
0,n1WpP7iowLc,2017-11-14,Eminem - Walk On Water (Audio) ft. Beyoncé,EminemVEVO,10,2017-11-10 17:00:03+00:00,"Eminem|""Walk""|""On""|""Water""|""Aftermath/Shady/In...",17158579,787425,43420,125882,https://i.ytimg.com/vi/n1WpP7iowLc/default.jpg,False,False,False,Eminem's new track Walk on Water ft. Beyoncé i...
1,0dBIkQ4Mz1M,2017-11-14,PLUSH - Bad Unboxing Fan Mail,iDubbbzTV,23,2017-11-13 17:00:00+00:00,"plush|""bad unboxing""|""unboxing""|""fan mail""|""id...",1014651,127794,1688,13030,https://i.ytimg.com/vi/0dBIkQ4Mz1M/default.jpg,False,False,False,STill got a lot of packages. Probably will las...
2,5qpjK5DgCt4,2017-11-14,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,2017-11-12 19:05:24+00:00,"racist superman|""rudy""|""mancuso""|""king""|""bach""...",3191434,146035,5339,8181,https://i.ytimg.com/vi/5qpjK5DgCt4/default.jpg,False,False,False,WATCH MY PREVIOUS VIDEO ▶ \n\nSUBSCRIBE ► http...
3,d380meD0W0M,2017-11-14,I Dare You: GOING BALD!?,nigahiga,24,2017-11-12 18:01:41+00:00,"ryan|""higa""|""higatv""|""nigahiga""|""i dare you""|""...",2095828,132239,1989,17518,https://i.ytimg.com/vi/d380meD0W0M/default.jpg,False,False,False,I know it's been a while since we did this sho...
4,2Vv-BfVoq4g,2017-11-14,Ed Sheeran - Perfect (Official Music Video),Ed Sheeran,10,2017-11-09 11:04:14+00:00,"edsheeran|""ed sheeran""|""acoustic""|""live""|""cove...",33523622,1634130,21082,85067,https://i.ytimg.com/vi/2Vv-BfVoq4g/default.jpg,False,False,False,🎧: https://ad.gt/yt-perfect\n💰: https://atlant...


Es importante saber que podemos realizar extracciones de cierta parte de la fecha, como el mes o el año y también podemos realizar operaciones aritméticas para sumar o restar días, meses, etc...


In [61]:
# extracción de partes de una fecha
canadian_youtube['year_publish_time'] = canadian_youtube['publish_time'].dt.year

# añadir dias a una fecha
canadian_youtube['trending_date_plus_seven_days'] = canadian_youtube['trending_date'] + pd.Timedelta(days=7)

# restar dias a una fecha
canadian_youtube['trending_date_less_ten_days'] = canadian_youtube['trending_date'] - pd.Timedelta(days=10)

# restar meses
canadian_youtube['trending_date_less_n_months'] = canadian_youtube['trending_date'] - pd.DateOffset(months=1)

# sumar dos días usando DateOffset
canadian_youtube['trending_date_plus_2_days_DateOffset'] = canadian_youtube['trending_date'] + pd.DateOffset(days=2)

# sumar veinte dos días usando DateOffset
canadian_youtube['trending_date_plus_20_days_DateOffset'] = canadian_youtube['trending_date'] + pd.DateOffset(days=20)
canadian_youtube

,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,...,comments_disabled,ratings_disabled,video_error_or_removed,description,year_publish_time,trending_date_plus_seven_days,trending_date_less_ten_days,trending_date_less_n_months,trending_date_plus_2_days_DateOffset,trending_date_plus_20_days_DateOffset
0,n1WpP7iowLc,2017-11-14,Eminem - Walk On Water (Audio) ft. Beyoncé,EminemVEVO,10,2017-11-10 17:00:03+00:00,"Eminem|""Walk""|""On""|""Water""|""Aftermath/Shady/In...",17158579,787425,43420,...,False,False,False,Eminem's new track Walk on Water ft. Beyoncé i...,2017,2017-11-21,2017-11-04,2017-10-14,2017-11-16,2017-12-04
1,0dBIkQ4Mz1M,2017-11-14,PLUSH - Bad Unboxing Fan Mail,iDubbbzTV,23,2017-11-13 17:00:00+00:00,"plush|""bad unboxing""|""unboxing""|""fan mail""|""id...",1014651,127794,1688,...,False,False,False,STill got a lot of packages. Probably will las...,2017,2017-11-21,2017-11-04,2017-10-14,2017-11-16,2017-12-04
2,5qpjK5DgCt4,2017-11-14,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,2017-11-12 19:05:24+00:00,"racist superman|""rudy""|""mancuso""|""king""|""bach""...",3191434,146035,5339,...,False,False,False,WATCH MY PREVIOUS VIDEO ▶ \n\nSUBSCRIBE ► http...,2017,2017-11-21,2017-11-04,2017-10-14,2017-11-16,2017-12-04
3,d380meD0W0M,2017-11-14,I Dare You: GOING BALD!?,nigahiga,24,2017-11-12 18:01:41+00:00,"ryan|""higa""|""higatv""|""nigahiga""|""i dare you""|""...",2095828,132239,1989,...,False,False,False,I know it's been a while since we did this sho...,2017,2017-11-21,2017-11-04,2017-10-14,2017-11-16,2017-12-04
4,2Vv-BfVoq4g,2017-11-14,Ed Sheeran - Perfect (Official Music Video),Ed Sheeran,10,2017-11-09 11:04:14+00:00,"edsheeran|""ed sheeran""|""acoustic""|""live""|""cove...",33523622,1634130,21082,...,False,False,False,🎧: https://ad.gt/yt-perfect\n💰: https://atlant...,2017,2017-11-21,2017-11-04,2017-10-14,2017-11-16,2017-12-04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40876,sGolxsMSGfQ,2018-06-14,HOW2: How to Solve a Mystery,Annoying Orange,24,2018-06-13 18:00:07+00:00,"annoying orange|""funny""|""fruit""|""talking""|""ani...",80685,1701,99,...,False,False,False,🚨 NEW MERCH! http://amzn.to/annoyingorange 🚨➤ ...,2018,2018-06-21,2018-06-04,2018-05-14,2018-06-16,2018-07-04
40877,8HNuRNi8t70,2018-06-14,Eli Lik Lik Episode 13 Partie 01,Elhiwar Ettounsi,24,2018-06-13 19:01:18+00:00,"hkayet tounsia|""elhiwar ettounsi""|""denya okhra...",103339,460,66,...,False,False,False,► Retrouvez vos programmes préférés : https://...,2018,2018-06-21,2018-06-04,2018-05-14,2018-06-16,2018-07-04
40878,GWlKEM3m2EE,2018-06-14,KINGDOM HEARTS III – SQUARE ENIX E3 SHOWCASE 2...,Kingdom Hearts,20,2018-06-11 17:30:53+00:00,"Kingdom Hearts|""KH3""|""Kingdom Hearts 3""|""Froze...",773347,25900,224,...,False,False,False,Find out more about Kingdom Hearts 3: https://...,2018,2018-06-21,2018-06-04,2018-05-14,2018-06-16,2018-07-04
40879,lbMKLzQ4cNQ,2018-06-14,Trump Advisor Grovels To Trudeau,The Young Turks,25,2018-06-13 04:00:05+00:00,"180612__TB02SorryExcuse|""News""|""Politics""|""The...",115225,2115,182,...,False,False,False,Peter Navarro isn’t talking so tough now. Ana ...,2018,2018-06-21,2018-06-04,2018-05-14,2018-06-16,2018-07-04


### ¿Qué es `unstack()`?

`unstack()` es un método de pandas que transforma un nivel del índice en columnas. Es muy útil después de un `groupby()` con múltiples niveles, porque convierte una salida en formato largo en una tabla comparativa (tipo pivote).




In [62]:
# ejemplo de unstack: filas: anio | columnas: canal | valores: número de videos
videos_por_anio_y_canal = (
    canadian_youtube.groupby(['year_publish_time', 'channel_title']).size().unstack(fill_value=0)
)

videos_por_anio_y_canal

channel_title,#AndresSTyle,#Mind Warehouse,#SeekingTheTruth,* Martyna *,- 欢迎订阅 -浙江卫视【奔跑吧】官方频道,-Wen Zhao Official文昭談古論今,078jordan1,0b1knob,10 MillionTM,10-Minutes Satisfaction,...,우찌,웃지 UTZI,이슈사건사고,이영애 (Lee Young - Ae),종합뉴스,창조영감클럽,타우TV,포스트쉐어,포크포크,활력소TV
year_publish_time,,,,,,,,,,,,,,,,,,,,,
2008,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2009,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2010,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2012,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2013,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2014,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2015,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2016,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2017,0,8,0,3,0,0,0,1,0,0,...,0,0,1,2,0,0,0,0,0,0


Una vez que sabemos extraer ciertas partes de una fecha se nos abre una posibilidad a realizar estudios de agrupación y filtrado por estas partes de la fecha, en especial, esto es muy útil para utilizar el año.


In [65]:
canadian_youtube_2017 = canadian_youtube.loc[canadian_youtube['publish_time'].dt.year==2017]

canadian_youtube.groupby(canadian_youtube['publish_time'].dt.year)['channel_title'].describe()

,count,unique,top,freq
publish_time,,,,
2008,5,2,lbisurfer,3
2009,1,1,KatieWear,1
2010,3,1,westsidewillz,3
2012,1,1,Josh McConnell,1
2013,9,5,SpikeySpotify,3
2014,6,6,Michelle 1,1
2015,7,6,TODAY’S TMJ4,2
2016,13,9,Unoriginal Content,2
2017,9900,1944,VikatanTV,49
